In [ ]:
# # NOTEBOOK NAME
# # McCoyWeather.ipynb
# # NOTEBOOK NAME

# # OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import matplotlib.cm as cm
import matplotlib.colors as mcolors

import matplotlib.dates as mdates
from scipy.ndimage import gaussian_filter
from scipy.interpolate import interp1d

In [ ]:
temps = pd.read_csv('/home/563/sg3241/McCoyTdailyMeans.csv')

In [ ]:
# CREATE A PERCENTILES ARRAY 365 DAY x 99 percentile

# Convert DateTime column from string to datetime (if not already done)
temps['DateTime'] = pd.to_datetime(temps['DateTime'])

# Remove leap days (29th February) before calculating day of year
temps_no_leap = temps[~((temps['DateTime'].dt.month == 2) & 
                         (temps['DateTime'].dt.day == 29))].copy()

# Extract day of year (now guaranteed to be 1-365 with no gaps)
temps_no_leap['DayOfYear'] = temps_no_leap['DateTime'].dt.dayofyear

# In leap years, days after 29 Feb are shifted by 1, so we correct for this
leap_years = temps_no_leap['DateTime'].dt.is_leap_year
after_feb = temps_no_leap['DateTime'].dt.dayofyear > 59  # After 29 Feb
temps_no_leap.loc[leap_years & after_feb, 'DayOfYear'] -= 1

# Initialize the array: 365 days × 99 percentiles
percentile_array = np.zeros((365, 99))

# Loop through each day of year (1 to 365)
for doy in range(1, 366):
    # Get all temperatures for this day of year across all years
    day_temps = temps_no_leap[temps_no_leap['DayOfYear'] == doy]['Mean'].values
    
    # Calculate percentiles 1 through 99
    if len(day_temps) > 0:
        percentiles = np.percentile(day_temps, np.arange(1, 100))
        percentile_array[doy - 1, :] = percentiles
    else:
        # If no data for this day, fill with NaN
        percentile_array[doy - 1, :] = np.nan

In [ ]:
# --- Smoothing variable ---
smooth_days = 0  # Number of days either side to include in rolling mean

# --- Apply circular (modular) rolling mean to percentile_array ---
# We use circular padding so that the smoothing wraps around 
# from 31 Dec back to 1 Jan and vice versa

def circular_rolling_mean(array, n_days):
    """
    Apply a circular rolling mean to a (365, 99) array along the day axis.
    n_days is the number of days either side of the centre day to include.
    Window size = 2 * n_days + 1
    """
    window = 2 * n_days + 1
    smoothed = np.zeros_like(array)

    # Pad the array circularly along the day axis (axis=0)
    padded = np.pad(array, ((n_days, n_days), (0, 0)), mode='wrap')

    # Calculate the rolling mean
    for d in range(365):
        smoothed[d, :] = padded[d:d + window, :].mean(axis=0)

    return smoothed

# Apply smoothing
percentile_array_smoothed = circular_rolling_mean(percentile_array, smooth_days)

In [ ]:
# CREATE A PERCENTILE CHART FOR THE FULL YEAR WITH FILL COLOURS

# --- Define custom colourmap control points ---
cmap_temps  = [-2,     7,      11,     15,           17.5,     24,       29,    35,          41,       50]
cmap_colors = ['cyan', 'blue', 'darkgreen', 'lightgreen', 'yellow', 'orange', 'red', 'darkred', 'magenta', 'pink']

# Build a normalised colourmap between temp_min and temp_max
# Convert named colours to RGB
rgb_colors = [mcolors.to_rgb(c) for c in cmap_colors]

# Create interpolators for R, G, B channels separately
norm_temps = [(t - cmap_temps[0]) / (cmap_temps[-1] - cmap_temps[0]) for t in cmap_temps]
r_interp = interp1d(norm_temps, [c[0] for c in rgb_colors], kind='linear')
g_interp = interp1d(norm_temps, [c[1] for c in rgb_colors], kind='linear')
b_interp = interp1d(norm_temps, [c[2] for c in rgb_colors], kind='linear')

def temp_to_color(T):
    """Convert a temperature value to an RGB colour using the custom colourmap."""
    T_norm = (T - cmap_temps[0]) / (cmap_temps[-1] - cmap_temps[0])
    T_norm = np.clip(T_norm, 0, 1)
    return (float(r_interp(T_norm)), 
            float(g_interp(T_norm)), 
            float(b_interp(T_norm)))

# --- Define temperature range ---
temp_min = np.floor(percentile_array_smoothed[:, 0].min()) - 5
temp_max = np.ceil(percentile_array_smoothed[:, 98].max()) + 5
temperatures = np.arange(temp_min, temp_max + 1, 1)

# --- For each temperature, find its percentile for each day ---
percentile_values = np.arange(1, 100)
reference_dates = pd.date_range('2001-01-01', periods=365, freq='D')

temp_percentile_lines = np.zeros((365, len(temperatures)))

for d in range(365):
    day_percentiles = percentile_array_smoothed[d, :]
    for t_idx, T in enumerate(temperatures):
        temp_percentile_lines[d, t_idx] = np.interp(T, day_percentiles, percentile_values,
                                                      left=0,    # <-- clamp to bottom
                                                      right=100) # <-- clamp to top

# --- Month boundary setup ---
month_starts_doy  = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335, 366]
month_starts_dates = [pd.Timestamp('2001-01-01') + pd.Timedelta(days=d - 1)
                      for d in month_starts_doy]
month_mids_doy    = [(month_starts_doy[i] + month_starts_doy[i + 1]) / 2
                      for i in range(12)]
month_mids_dates  = [pd.Timestamp('2001-01-01') + pd.Timedelta(days=d - 1)
                      for d in month_mids_doy]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# --- Create the plot ---
fig, ax = plt.subplots(figsize=(14, 8))

# --- Fill between consecutive temperature lines ---
for t_idx in range(len(temperatures) - 1):
    T_low  = temperatures[t_idx]
    T_high = temperatures[t_idx + 1]
    T_mid  = (T_low + T_high) / 2.0
    fill_color = temp_to_color(T_mid)

    y1 = temp_percentile_lines[:, t_idx]
    y2 = temp_percentile_lines[:, t_idx + 1]

    # Mask out NaN values
    valid = ~(np.isnan(y1) | np.isnan(y2))
    if valid.any():
        ax.fill_between(reference_dates, 
                        np.where(valid, y1, np.nan),
                        np.where(valid, y2, np.nan),
                        color=fill_color, alpha=1.0, linewidth=0)

# --- Plot temperature lines on top ---
for t_idx, T in enumerate(temperatures):
    T_int = int(round(T))
    if T_int % 10 == 0:
        lw = 2.5
    elif T_int % 5 == 0:
        lw = 1.2
    else:
        lw = 0.4

    ax.plot(reference_dates, temp_percentile_lines[:, t_idx],
            color='black', linewidth=lw, alpha=0.8)

# --- x-axis formatting ---
ax.set_xticks(month_mids_dates)
ax.set_xticklabels(month_names)
ax.tick_params(axis='x', length=0)

# --- x-axis vertical month boundary lines ---
quarter_months = {3, 6, 9, 12}
for i, date in enumerate(month_starts_dates):
    month_num = i + 1
    if month_num in quarter_months or month_num == 1:
        lw_vline = 1.2
    else:
        lw_vline = 0.6
    ax.axvline(date, color='white', linewidth=lw_vline, zorder=5)

# --- y-axis horizontal percentile grid lines ---
for percentile in range(10, 100, 10):
    if percentile == 50:
        lw_hline = 2.0
    else:
        lw_hline = 0.6
    ax.axhline(percentile, color='white', linewidth=lw_hline, zorder=5)


# --- y-axis formatting ---
ax.set_ylim(1, 99)
ax.set_yticks(np.arange(10, 100, 10))
ax.set_ylabel('Percentile')

# --- x-axis limits ---
ax.set_xlim(pd.Timestamp('2001-01-01'), pd.Timestamp('2001-12-31'))

ax.set_title('Temperature Percentile by Day of Year')
plt.tight_layout()
plt.show()



In [ ]:
# count days below 10 C

# Convert DateTime column from string to datetime (if not already done)
temps['DateTime'] = pd.to_datetime(temps['DateTime'])

# Filter for temperatures below 10°C
below_10 = temps[temps['Mean'] < 10].copy()

# Sort by DateTime to ensure consecutive days are in order
below_10 = below_10.sort_values('DateTime').reset_index(drop=True)

# Calculate the difference in days between consecutive rows
below_10['DayDiff'] = below_10['DateTime'].diff().dt.days

# Create a group identifier for consecutive day blocks
# A new block starts when the day difference is not 1 (or is NaN for the first row)
below_10['Block'] = (below_10['DayDiff'] != 1).fillna(True).cumsum()

# Group by block and get the first DateTime and count of days in each block
result = below_10.groupby('Block').agg(
    DateTime=('DateTime', 'first'),
    DaysCount=('DateTime', 'count')
).reset_index(drop=True)

print(result)

In [ ]:
result_sorted = result.sort_values('DaysCount').reset_index(drop=True)

In [ ]:
plt.hist(result_sorted['DaysCount'][:],bins=np.arange(0.5, 10.5, 1))

In [ ]:
result_sorted 

In [ ]:

# Convert DateTime column from string to datetime
temps['DateTime'] = pd.to_datetime(temps['DateTime'])
temps['Year'] = temps['DateTime'].dt.year

# Normalise all dates to the same dummy year (2000 - a leap year)
temps['NormalisedDate'] = temps['DateTime'].apply(
    lambda dt: dt.replace(year=2000)
)

# Set up blue-to-red colourmap across years
years = sorted(temps['Year'].unique())
norm = mcolors.Normalize(vmin=min(years), vmax=max(years))
cmap = cm.coolwarm

# Create the plot
fig, ax = plt.subplots(figsize=(13, 6))

# Plot each year with a colour from the colourmap
for year in years:
    year_data = temps[temps['Year'] == year]
    colour = cmap(norm(year))
    ax.plot(year_data['NormalisedDate'], year_data['Mean'],
            color=colour, marker='o', markersize=3, alpha=0.7)

# --- X-axis: ticks and grid at month borders, labels in the middle ---

# Ticks and grid lines at the 1st of each month (+ Jan of next year to close off Dec)
month_borders = [pd.Timestamp(f'2000-{m:02d}-01') for m in range(1, 13)] + \
                [pd.Timestamp('2001-01-01')]
ax.set_xticks(month_borders)
ax.set_xticklabels([])  # Hide the border tick labels

# Add month name labels manually at the midpoint of each month
mid_month_dates = [pd.Timestamp(f'2000-{m:02d}-15') for m in range(1, 13)]
for mid in mid_month_dates:
    ax.text(mid, -0.015, mid.strftime('%b'),
            ha='center', va='top', transform=ax.get_xaxis_transform())

# Grid lines at month borders only
ax.xaxis.grid(True, alpha=0.3)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Set x-axis limits
ax.set_xlim(pd.Timestamp('2000-01-01'), pd.Timestamp('2001-01-01'))

# --- Colourbar legend outside the plot on the right ---
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Required dummy call
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Year')
cbar.set_ticks(years)
cbar.set_ticklabels([str(y) for y in years])

plt.ylabel('°C')
plt.title('Mean Daily Temperature')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:

# Convert DateTime column from string to datetime
temps['DateTime'] = pd.to_datetime(temps['DateTime'])
temps['Year'] = temps['DateTime'].dt.year

# Normalise all dates to the same dummy year (2000 - a leap year)
temps['NormalisedDate'] = temps['DateTime'].apply(
    lambda dt: dt.replace(year=2000)
)

# Convert NormalisedDate to numeric for binning
temps['DateNumeric'] = temps['NormalisedDate'].astype(np.int64)

# Define bin edges
n_date_bins = 366
n_temp_bins = 100

date_bins = np.linspace(
    pd.Timestamp('2000-01-01').value,
    pd.Timestamp('2001-01-01').value,
    n_date_bins + 1
)

# Pad the temp range so Gaussian blur can fade out smoothly at the edges
temp_padding = 3
temp_bins = np.linspace(
    temps['Mean'].min() - temp_padding,
    temps['Mean'].max() + temp_padding,
    n_temp_bins + 1
)

# Compute 2D histogram
density, _, _ = np.histogram2d(
    temps['DateNumeric'], temps['Mean'],
    bins=[date_bins, temp_bins]
)
density = density.T  # Transpose so rows = temp, cols = date

# Apply Gaussian blur to create smooth density appearance
density = gaussian_filter(density, sigma=[2, 2])

# Only mask truly empty bins
density[density == 0] = np.nan

# Convert bin edges back to Timestamps for the x-axis
date_bin_edges = pd.to_datetime(date_bins)

# Create the plot with black background
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')

mesh = ax.pcolormesh(
    date_bin_edges, temp_bins, density,
    cmap='nipy_spectral', shading='flat', zorder=1
)

# Colourbar for density
cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label('Density', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

# --- Layered grid lines (zorder=2 to sit on top of mesh) ---

temp_min = int(np.floor(temps['Mean'].min() - temp_padding))
temp_max = int(np.ceil(temps['Mean'].max() + temp_padding))
all_int_degrees = np.arange(temp_min, temp_max + 1)

# Thin lines for all integer degrees
for t in all_int_degrees:
    ax.axhline(t, color='white', linewidth=0.3, alpha=0.3, zorder=2)

# Medium lines for 5, 15, 25, ... (odd multiples of 5)
for t in all_int_degrees:
    if t % 10 == 5 or t % 10 == -5:
        ax.axhline(t, color='white', linewidth=0.8, alpha=0.5, zorder=2)

# Thick lines for every 10 degrees (0, 10, 20, ...)
for t in all_int_degrees:
    if t % 10 == 0:
        ax.axhline(t, color='white', linewidth=1.5, alpha=0.7, zorder=2)

# Vertical grid lines at month borders
month_borders = [pd.Timestamp(f'2000-{m:02d}-01') for m in range(1, 13)] + \
                [pd.Timestamp('2001-01-01')]
quarter_months = {1, 4, 7, 10}  # Jan, Apr, Jul, Oct

for i, mb in enumerate(month_borders):
    month_num = mb.month if mb.year == 2000 else 1  # handle 2001-01-01
    if month_num in quarter_months:
        ax.axvline(mb, color='white', linewidth=1.5, alpha=0.7, zorder=2)
    else:
        ax.axvline(mb, color='white', linewidth=0.8, alpha=0.5, zorder=2)

# --- X-axis labels in the middle of each month ---
ax.set_xticks(month_borders)
ax.set_xticklabels([])

mid_month_dates = [pd.Timestamp(f'2000-{m:02d}-15') for m in range(1, 13)]
for mid in mid_month_dates:
    ax.text(mid, -0.015, mid.strftime('%b'),
            ha='center', va='top', transform=ax.get_xaxis_transform(),
            color='white')

ax.set_xlim(pd.Timestamp('2000-01-01'), pd.Timestamp('2001-01-01'))

# Style axes for black background
ax.tick_params(colors='white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')

plt.ylabel('°C')
plt.title('Mean Daily Temperature — Density')
plt.tight_layout()
plt.show()


In [ ]:
# CHAD FUNCTION TO STORE TRAILING MEANS

def consecutive_rolling_mean(df, days, date_col='DateTime', value_col='Mean'):
    """
    Calculates a rolling mean over `days` consecutive days.
    Only calculates a value if all `days` preceding days (inclusive) have data.
    Returns a new DataFrame with the last day as the date and the rolling mean.

    Parameters
    ----------
    df   : DataFrame containing date and value columns
    days : int, number of consecutive days to average over
    """

    # Ensure sorted by date and work on a clean copy
    df = df.copy().sort_values(date_col).reset_index(drop=True)

    # Normalise to date only (no time component) for gap checking
    df['_date'] = pd.to_datetime(df[date_col]).dt.normalize()

    results = []

    for i in range(len(df)):
        # We need `days` rows ending at row i
        if i < days - 1:
            continue

        window = df.iloc[i - days + 1 : i + 1]

        # Check all dates in the window are consecutive (no gaps)
        date_diffs = window['_date'].diff().dropna()
        all_consecutive = (date_diffs == pd.Timedelta(days=1)).all()

        if all_consecutive:
            results.append({
                date_col: df.loc[i, date_col],
                f'{days}-Day Mean': window[value_col].mean()
            })

    result_df = pd.DataFrame(results).reset_index(drop=True)
    return result_df

In [ ]:
Means3 = consecutive_rolling_mean(temps,3)
Means5 = consecutive_rolling_mean(temps,5)
Means7 = consecutive_rolling_mean(temps,7)
Means10 = consecutive_rolling_mean(temps,10)

In [ ]:
temps_sorted = temps.sort_values('Mean').reset_index(drop=True)


In [ ]:
Means5_sorted = Means5.sort_values('7-Day Mean').reset_index(drop=True)

In [ ]:
Means7_sorted

In [ ]:
fig, ax = plt.subplots()

ax.hist(temps['Mean'], bins=np.arange(0, 41, 1),
        color='orange', edgecolor='red', linewidth=0.8,
        rwidth=0.85)  # rwidth < 1 adds a small gap between bars

# Grid every 5 degrees (vertical lines)
ax.set_xticks(np.arange(0, 41, 5))
ax.xaxis.grid(True, alpha=0.5)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

ax.set_xlabel('Mean Temperature (°C)')
ax.set_ylabel('Number of Days')
ax.set_title('Distribution of Mean Daily Temperature')

plt.xlim([0,40])

plt.tight_layout()
plt.show()



In [ ]:
fig, ax = plt.subplots()

ax.hist(Means3['3-Day Mean'], bins=np.arange(0, 41, 1),
        color='yellow', edgecolor='orange', linewidth=0.8,
        rwidth=0.85)  # rwidth < 1 adds a small gap between bars

# Grid every 5 degrees (vertical lines)
ax.set_xticks(np.arange(0, 41, 5))
ax.xaxis.grid(True, alpha=0.5)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

ax.set_xlabel('Mean Temperature (°C)')
ax.set_ylabel('Number of Days')
ax.set_title('Distribution of 3-Day Mean Daily Temperature')

plt.xlim([0,40])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.hist(Means5['5-Day Mean'], bins=np.arange(0, 41, 1),
        color='green', edgecolor='blue', linewidth=0.8,
        rwidth=0.85)  # rwidth < 1 adds a small gap between bars

# Grid every 5 degrees (vertical lines)
ax.set_xticks(np.arange(0, 41, 5))
ax.xaxis.grid(True, alpha=0.5)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

ax.set_xlabel('Mean Temperature (°C)')
ax.set_ylabel('Number of Days')
ax.set_title('Distribution of 5-Day Mean Daily Temperature')

plt.xlim([0,40])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.hist(Means7['7-Day Mean'], bins=np.arange(0, 41, 1),
        color='cyan', edgecolor='blue', linewidth=0.8,
        rwidth=0.85)  # rwidth < 1 adds a small gap between bars

# Grid every 5 degrees (vertical lines)
ax.set_xticks(np.arange(0, 41, 5))
ax.xaxis.grid(True, alpha=0.5)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

ax.set_xlabel('Mean Temperature (°C)')
ax.set_ylabel('Number of Days')
ax.set_title('Distribution of 7-Day Mean Daily Temperature')

plt.xlim([0,40])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.hist(Means10['10-Day Mean'], bins=np.arange(0, 41, 1),
        color='purple', edgecolor='blue', linewidth=0.8,
        rwidth=0.85)  # rwidth < 1 adds a small gap between bars

# Grid every 5 degrees (vertical lines)
ax.set_xticks(np.arange(0, 41, 5))
ax.xaxis.grid(True, alpha=0.5)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

ax.set_xlabel('Mean Temperature (°C)')
ax.set_ylabel('Number of Days')
ax.set_title('Distribution of 10-Day Mean Daily Temperature')

plt.xlim([0,40])

plt.tight_layout()
plt.show()